# GRPO post-training for a pretrained Decision Transformer

This notebook shows how to run **Group Relative Policy Optimization (GRPO)** as an online post-training stage on top of a pretrained household Decision Transformer.

## Prerequisites
- Household raw CSVs available under `data/household/raw/`.
- A pretrained DT checkpoint, for example `models/household/dt/dt_model.pt`.
- If you need to create the checkpoint first, run `python3 src/pretrain_decision_transformer.py ...` from the repo root.


In [ ]:
from pathlib import Path
import os
import sys

def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'README.md').exists() and (candidate / 'src').exists():
            return candidate
    return start

REPO_ROOT = _find_repo_root(Path.cwd().resolve())
os.chdir(REPO_ROOT)
for _extra_path in (REPO_ROOT, REPO_ROOT / 'src'):
    _extra_str = str(_extra_path)
    if _extra_str not in sys.path:
        sys.path.insert(0, _extra_str)

REPO_ROOT


In [ ]:
import json
import numpy as np
import polars as pl
import torch

from EnergySimEnv import SolarBatteryEnv
from decision import Agent
from grpo_posttraining import GRPOPrompt, GRPOTrainer, load_pretrained_dt_for_grpo
from helper import transform_polars_df

device = 'cuda' if torch.cuda.is_available() else 'cpu'
device


In [ ]:
RAW_DATA_PATH = REPO_ROOT / 'data' / 'household' / 'raw' / '2010-2011 Solar home electricity data.csv'
MODEL_KWARGS_PATH = REPO_ROOT / 'models' / 'household' / 'dt' / 'decision_transformer_model_kwargs.json'
CHECKPOINT_PATH = REPO_ROOT / 'models' / 'household' / 'dt' / 'dt_model.pt'
GRPO_OUTPUT_PATH = REPO_ROOT / 'models' / 'household' / 'dt' / 'dt_model_grpo.pt'

if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(f'Missing household raw CSV: {RAW_DATA_PATH}')
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f'Missing pretrained DT checkpoint: {CHECKPOINT_PATH}')

RAW_DATA_PATH, CHECKPOINT_PATH


In [ ]:
raw_df = pl.read_csv(RAW_DATA_PATH, skip_rows=1)
customer_id = raw_df['Customer'].unique().sort()[0]
customer_df = raw_df.filter(pl.col('Customer') == customer_id)
transformed_df = transform_polars_df(
    customer_df,
    import_energy_price=0.23,
    export_energy_price=0.015,
    price_periods='7am – 10am | 4pm – 9pm',
    default_import_energy_price=0.15,
    default_export_energy_price=0.01,
)

transformed_df.head()


In [ ]:
def make_env():
    return SolarBatteryEnv(
        transformed_df,
        battery_capacity=13.5,
        max_battery_flow=5.0,
        init_battery_level=6.75,
        max_step=min(len(transformed_df), 24 * 12),
    )

with open(MODEL_KWARGS_PATH, 'r', encoding='utf-8') as fh:
    model_kwargs = json.load(fh)

model, reference_model = load_pretrained_dt_for_grpo(
    model_kwargs,
    CHECKPOINT_PATH,
    device=device,
)
model.return_scale


In [ ]:
def evaluate_dt_policy(dt_model, episodes=3, rtg_value=0.0):
    rewards = []
    for seed in range(episodes):
        env = make_env()
        agent = Agent(env, algorithm='dt', model=dt_model, rtg_value=rtg_value, reset_seed=seed)
        logs = agent.run_episode(render=False, display_progress=False)
        rewards.append(sum(float(step['reward']) for step in logs))
    return rewards

baseline_rewards = evaluate_dt_policy(model, episodes=3, rtg_value=0.0)
baseline_rewards


In [ ]:
prompts = [GRPOPrompt(seed=seed, rtg_value=0.0) for seed in range(4)]
trainer = GRPOTrainer(
    model,
    reference_model=reference_model,
    device=device,
    lr=1e-5,
    clip_ratio=0.2,
    kl_coeff=0.02,
    initial_log_std=-1.0,
    trainable_log_std=True,
)

history = trainer.train(
    make_env,
    prompts=prompts,
    iterations=3,
    group_size=4,
    update_epochs=2,
    minibatch_size=32,
    dt_gamma=0.99,
)
history


In [ ]:
post_grpo_rewards = evaluate_dt_policy(model, episodes=3, rtg_value=0.0)
print('Baseline rewards:', baseline_rewards)
print('Post-GRPO rewards:', post_grpo_rewards)
print('Mean improvement:', float(np.mean(post_grpo_rewards) - np.mean(baseline_rewards)))


In [ ]:
torch.save(model.state_dict(), GRPO_OUTPUT_PATH)
print(f'Saved GRPO-updated DT weights to {GRPO_OUTPUT_PATH}')


## Notes
- GRPO here uses group-normalized episode returns as advantages.
- The reference model anchors updates with a KL penalty so the post-trained policy stays close to the pretrained DT.
- Increase the number of prompts, group size, or iterations once the smoke run is stable in your runtime.
